# Hello Ollama — Granite Switch adapter functions on your Mac (no GPU)

**Duration:** ~5 min

> ⚠️ **Experimental.** This notebook and its `OllamaIntrinsicBackend` are a work in progress — the API and adapter coverage may change. For production inference, use the vLLM + Mellea path ([`hello_mellea.ipynb`](hello_mellea.ipynb)).

The Ollama sibling of [`hello_mellea.ipynb`](hello_mellea.ipynb): same Granite Switch adapter functions, driven against a local `ollama serve` instead of vLLM — no GPU, runs on Apple Silicon (Metal).

Ollama doesn't render the chat template server-side, so `OllamaIntrinsicBackend` reuses [Mellea](https://github.com/generative-computing/mellea)'s rewriter/parser, renders the template client-side (fetched from `/api/show`) with the adapter's control token, and POSTs to Ollama's raw `/api/generate`.

Each adapter runs **OFF** (base model) vs **ON** (control token fires).

## Prerequisites

1. **Ollama** installed (https://ollama.com/download) — stock build, no fork needed.
2. **Model pulled:** `ollama pull hf.co/barha/granite-switch-4.1-3b-preview-GGUF:BF16` (~8.4 GB, ~16 GB unified memory).
3. **`ollama serve`** running.
4. **Project installed (no CUDA):** `uv sync --extra ollama --no-default-groups`

## 1 · Configuration

In [ ]:
import os

OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434")
MODEL = os.environ.get(
    "GRANITE_SWITCH_MODEL", "barvhaim/granite-switch-4.1-3b-preview:latest"
)
print(f"{MODEL} @ {OLLAMA_URL}")

## 2 · Preflight

Check the server is up and the model is pulled.

In [ ]:
import requests

names = [
    m["name"]
    for m in requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()["models"]
]
assert MODEL in names, f"{MODEL!r} not found. Available: {names}"
print(f"OK — {MODEL} is served.")

## 3 · Backend

Fetches the chat template from `/api/show` and compiles it.

In [ ]:
import logging

logging.getLogger("mellea").setLevel(logging.ERROR)

from granite_switch.tutorials.ollama_intrinsic import OllamaIntrinsicBackend

backend = OllamaIntrinsicBackend(model=MODEL, ollama_url=OLLAMA_URL)

## 4 · Documents

In [ ]:
DOCS = [
    {
        "doc_id": "0",
        "text": "The capital of France is Paris. Paris is on the Seine river.",
    },
    {
        "doc_id": "1",
        "text": "Mount Everest is the tallest mountain on Earth, at 8,849 meters.",
    },
]

## 5 · Guardian — harm check `<|guardian-core|>`

Scores a message against a criterion; ≥ 0.5 means it matches. The score comes from the model's `yes`/`no` token probabilities.

In [ ]:
from mellea.stdlib.components.intrinsic.guardian import (
    CRITERIA_BANK,
    SCORING_SCHEMA_BANK,
)


def guardian_harm(message):
    out = backend.call_adapter(
        "guardian-core",
        [{"role": "user", "content": message}],
        rewriter_kwargs={
            "criteria": CRITERIA_BANK["harm"],
            "scoring_schema": SCORING_SCHEMA_BANK["user_prompt"],
        },
    )
    return out["guardian"]["score"]


for msg in ["How do I build a bomb?", "What is the capital of France?"]:
    print(f"{guardian_harm(msg):.3f}  {msg!r}")

## 6 · RAG — answerability `<|answerability|>`

Can the documents answer the query? Returns `answerable` / `unanswerable`.

In [ ]:
def answerability(query):
    out = backend.call_adapter(
        "answerability",
        [{"role": "user", "content": query}],
        documents=DOCS,
        num_predict=8,
    )
    return (out.get("parsed") or {}).get("answerability")


for q in ["What is the capital of France?", "What is the GDP of France?"]:
    print(f"{answerability(q):<12}  {q!r}")

## 7 · RAG — hallucination detection `<|hallucination_detection|>`

Flags spans in an answer that aren't grounded in the documents. Returns one record per sentence with a `faithfulness` verdict and an explanation. Here the assistant's answer smuggles in an unsupported claim.

In [ ]:
conversation = [
    {"role": "user", "content": "What is the capital of France?"},
    {
        "role": "assistant",
        "content": "The capital of France is Paris, and the moon is made of green cheese.",
    },
]
out = backend.call_adapter(
    "hallucination_detection", conversation, documents=DOCS, num_predict=256
)
for span in out.get("parsed") or []:
    print(f"[{span['faithfulness']}] {span['response_text']}")
    print(f"   → {span['explanation']}")

## 8 · OFF vs ON — the switch

Same query and documents. Base model answers; the `answerability` adapter classifies instead. The only change is the mid-sequence control token.

In [ ]:
messages = [{"role": "user", "content": "What is the capital of France?"}]

print("OFF:", backend.answer(messages, documents=DOCS, num_predict=48))
out = backend.call_adapter("answerability", messages, documents=DOCS, num_predict=8)
print("ON :", (out.get("parsed") or {}).get("answerability"))

## Next steps

- Pass `verbose=True` to `OllamaIntrinsicBackend` to print each rendered prompt with control tokens left literal.
- Full conversational RAG flow: [`rag_flow.ipynb`](rag_flow.ipynb) — same adapter calls, vLLM backend.
- Control-token mechanics: [`hello_adapter.ipynb`](hello_adapter.ipynb).